<a href="https://colab.research.google.com/github/[USERNAME]/northstar-analytics-cw1/blob/main/notebooks/04_mongodb_design_crud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04 — MongoDB Atlas: Schema Design and CRUD Operations

**NorthStar Analytics — Coursework 1, Section 6**

This notebook builds the four NorthStar collections (`service_cases`, `drivers`, `vehicles`, `app_events`) on MongoDB Atlas and demonstrates the full CRUD repertoire from Lecture 10 plus the aggregation patterns from Lecture 11.

**Atlas setup:** create a free M0 cluster, add a database user, allow your IP, copy the connection string, and store it in Colab as a Secret named `MONGO_URI`.

In [ ]:
!pip install pymongo dnspython -q

from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError
from datetime import datetime, timezone
import pandas as pd
import numpy as np

In [ ]:
# Load Atlas URI from Colab Secret (recommended)
from google.colab import userdata
MONGO_URI = userdata.get('MONGO_URI')

client = MongoClient(MONGO_URI)
db = client['northstar']

print(f'Connected. Server version: {client.server_info()["version"]}')
print(f'Existing collections: {db.list_collection_names()}')

## 1. Load and prepare data

In [ ]:
customers  = pd.read_csv('customers.csv',  parse_dates=['signup_date'])
orders     = pd.read_csv('orders.csv',     parse_dates=['order_created_at'])
deliveries = pd.read_csv('deliveries.csv',
                         parse_dates=['dispatch_time','delivery_completed_at'])
drivers    = pd.read_csv('drivers.csv')
vehicles   = pd.read_csv('vehicles.csv',   parse_dates=['commission_date'])
incidents  = pd.read_csv('incidents.csv',  parse_dates=['reported_at'])
complaints = pd.read_csv('complaints.csv', parse_dates=['created_at'])
app_events = pd.read_csv('app_events.csv', parse_dates=['event_timestamp'])

def normalise_zone(s):
    if pd.isna(s): return None
    mapping = {'ctr':'Central','central':'Central','airport':'Airport',
               'north':'North','south':'South','east':'East',
               'west':'West','riverside':'Riverside'}
    s = str(s).strip().lower()
    return mapping.get(s, s.title())

for col in ['pickup_zone', 'dropoff_zone']:
    orders[col] = orders[col].apply(normalise_zone)
customers['home_zone']    = customers['home_zone'].apply(normalise_zone)
drivers['base_zone']      = drivers['base_zone'].apply(normalise_zone)
vehicles['assigned_zone'] = vehicles['assigned_zone'].apply(normalise_zone)

# Anomaly removal
deliveries['actual_duration_hrs'] = (
    deliveries['delivery_completed_at'] - deliveries['dispatch_time']
).dt.total_seconds() / 3600
n_before = len(deliveries)
deliveries = deliveries[
    (deliveries['actual_duration_hrs'].isna()) |
    (deliveries['actual_duration_hrs'] >= 0)
].copy()
print(f'Anomalies removed: {n_before - len(deliveries)}')

## 2. Build service_cases documents (the operational aggregate)

In [ ]:
def to_dt(v):
    return None if pd.isna(v) else pd.Timestamp(v).to_pydatetime()

def to_native(v):
    if pd.isna(v): return None
    if hasattr(v, 'item'): return v.item()
    return v

def build_service_case(order, delivery, customer,
                       incidents_for_delivery, complaints_for_order):
    timeline = []
    if not pd.isna(delivery['dispatch_time']):
        timeline.append({'ts': to_dt(delivery['dispatch_time']), 'event': 'dispatch'})
    for inc in incidents_for_delivery:
        if inc.get('reported_at'):
            timeline.append({'ts': inc['reported_at'],
                             'event': 'incident_logged',
                             'detail': inc.get('type')})
    if not pd.isna(delivery['delivery_completed_at']):
        timeline.append({'ts': to_dt(delivery['delivery_completed_at']),
                         'event': 'completion',
                         'detail': delivery['delivery_status']})
    for c in complaints_for_order:
        if c.get('created_at'):
            timeline.append({'ts': c['created_at'],
                             'event': 'complaint_lodged',
                             'detail': c.get('type')})
    timeline = [t for t in timeline if t['ts'] is not None]
    timeline.sort(key=lambda t: t['ts'])

    return {
        'case_id': f"CASE-{order['order_id']}",
        'created_at': to_dt(order['order_created_at']),
        'service_type': order['service_type'],
        'priority_level': order['priority_level'],
        'order_value': float(order['order_value']),
        'booking_channel': order['booking_channel'],
        'special_handling_flag': bool(order['special_handling_flag']),
        'customer': {
            'customer_id': customer['customer_id'],
            'customer_type': customer['customer_type'],
            'home_zone': customer['home_zone'],
            'loyalty_score': float(customer['loyalty_score']),
            'app_engagement_score': float(customer['app_engagement_score']),
            'account_status': customer['account_status']
        },
        'route': {
            'pickup_zone': order['pickup_zone'],
            'dropoff_zone': order['dropoff_zone'],
            'promised_window_hours': int(order['promised_window_hours'])
        },
        'delivery': {
            'delivery_id': delivery['delivery_id'],
            'driver_ref': delivery['driver_id'],
            'vehicle_ref': delivery['vehicle_id'],
            'hub_id': delivery['hub_id'],
            'dispatch_time': to_dt(delivery['dispatch_time']),
            'completed_at': to_dt(delivery['delivery_completed_at']),
            'status': delivery['delivery_status'],
            'route_distance_km': float(delivery['route_distance_km']),
            'manual_override_count': int(delivery['manual_route_override_count']),
            'proof_missing': bool(delivery['proof_of_completion_missing']),
            'customer_rating': to_native(delivery.get('customer_rating_post_delivery')),
            'fuel_cost': float(delivery['fuel_or_charge_cost'])
        },
        'incidents': incidents_for_delivery,
        'complaints': complaints_for_order,
        'event_timeline': timeline,
        'metadata': {
            'last_updated': datetime.now(timezone.utc),
            'schema_version': '1.0',
            'source_system': 'NorthStar_ETL_2026'
        }
    }

documents = []
for _, order in orders.iterrows():
    d = deliveries[deliveries['order_id'] == order['order_id']]
    if d.empty: continue
    delivery = d.iloc[0]
    cu = customers[customers['customer_id'] == order['customer_id']]
    if cu.empty: continue
    customer = cu.iloc[0]
    inc = incidents[incidents['delivery_id'] == delivery['delivery_id']]
    inc_docs = [{'incident_id': r['incident_id'],
                 'type': r['incident_type'],
                 'reported_at': to_dt(r['reported_at']),
                 'severity': r['severity'],
                 'resolution_status': r['resolution_status'],
                 'resolved_hours': to_native(r['resolved_hours'])}
                for _, r in inc.iterrows()]
    co = complaints[complaints['order_id'] == order['order_id']]
    co_docs = [{'complaint_id': r['complaint_id'],
                'type': r['complaint_type'],
                'channel': r['channel'],
                'severity': r['severity'],
                'status': r['status'],
                'created_at': to_dt(r['created_at']),
                'resolution_days': to_native(r['resolution_days']),
                'compensation_amount': to_native(r['compensation_amount'])}
               for _, r in co.iterrows()]
    documents.append(build_service_case(order, delivery, customer, inc_docs, co_docs))

print(f'Built {len(documents)} service_cases documents')

## 3. CREATE — bulk insert

In [ ]:
cases = db['service_cases']
cases.drop()  # idempotent rebuild

try:
    result = cases.insert_many(documents, ordered=False)
    print(f'Inserted {len(result.inserted_ids)} service_cases')
except BulkWriteError as bwe:
    print(f'Bulk write issues: {bwe.details}')

## 4. READ — operational queries

These queries use the operators from Lecture 10: `$or`, `$in`, `$gt`, `$gte`, embedded-document queries with dot notation, and cursor methods (`limit`, `sort`).

In [ ]:
# READ 1 — all failed cases in the Central zone, hot-first by override count
failed_central = list(cases.find(
    {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'},
    {'case_id': 1, 'service_type': 1, 'customer.customer_id': 1,
     'delivery.driver_ref': 1, 'delivery.manual_override_count': 1, '_id': 0}
).sort('delivery.manual_override_count', DESCENDING).limit(20))

print(f'Central failed cases: {len(failed_central)}')
for c in failed_central[:5]:
    print(c)

In [ ]:
# READ 2 — high-priority OR Medical service, with $or + $in
recent_priority = list(cases.find({
    '$or': [
        {'priority_level': {'$in': ['Critical', 'High']}},
        {'service_type': 'Medical'}
    ]
}, {'case_id': 1, 'service_type': 1, 'priority_level': 1,
    'delivery.status': 1, '_id': 0}).limit(50))

print(f'High priority or Medical cases: {len(recent_priority)}')
for c in recent_priority[:5]:
    print(c)

In [ ]:
# READ 3 — embedded document query for any open Damage complaint
damage_open = list(cases.find(
    {'complaints.type': 'Damage', 'complaints.status': 'Open'},
    {'case_id': 1, 'customer.customer_id': 1, 'complaints.$': 1, '_id': 0}
))

print(f'Open damage complaints: {len(damage_open)}')
for c in damage_open[:3]:
    print(c)

## 5. UPDATE — resolve a complaint with positional operator

Demonstrates `update_one`, `$set` on an embedded array element using the `$` positional operator, and `$push` to append to an array (Lecture 10).

In [ ]:
# Find a real complaint to update
sample = cases.find_one({'complaints': {'$ne': []}})
case_id = sample['case_id']
complaint_id = sample['complaints'][0]['complaint_id']
print(f'Updating {case_id} complaint {complaint_id}')

result = cases.update_one(
    {'case_id': case_id, 'complaints.complaint_id': complaint_id},
    {
        '$set': {
            'complaints.$.status': 'Resolved',
            'complaints.$.resolution_days': 7,
            'metadata.last_updated': datetime.now(timezone.utc)
        },
        '$push': {
            'event_timeline': {
                'ts': datetime.now(timezone.utc),
                'event': 'complaint_resolved',
                'detail': complaint_id
            }
        }
    }
)
print(f'Matched {result.matched_count}, modified {result.modified_count}')

## 6. DELETE — defensive cleanup

In [ ]:
result = cases.delete_many({'order_value': {'$lte': 0}})
print(f'Deleted {result.deleted_count} invalid records')

## 7. Aggregation pipelines (Lecture 11)

Three pipelines using `$match`, `$group`, `$unwind`, `$lookup`, `$project`, `$sort`, `$limit`, `$addFields`, accumulators (`$sum`, `$avg`, `$first`, `$last`).

In [ ]:
# Pipeline 1 — failure rate by zone × service_type
p1 = [
    {'$match': {'delivery.status': {'$in': ['OnTime', 'Delayed', 'Failed']}}},
    {'$group': {
        '_id': {'zone': '$route.pickup_zone', 'service': '$service_type'},
        'n_total': {'$sum': 1},
        'n_failed': {'$sum': {
            '$cond': [{'$eq': ['$delivery.status', 'Failed']}, 1, 0]}},
        'avg_rating': {'$avg': '$delivery.customer_rating'}
    }},
    {'$addFields': {
        'failure_pct': {'$round': [
            {'$multiply': [{'$divide': ['$n_failed', '$n_total']}, 100]}, 2]},
        'avg_rating': {'$round': ['$avg_rating', 3]}
    }},
    {'$sort': {'failure_pct': -1}},
    {'$limit': 10}
]
for d in cases.aggregate(p1): print(d)

In [ ]:
# Pipeline 2 — repeat complainants using $unwind
p2 = [
    {'$match': {'complaints': {'$exists': True, '$ne': []}}},
    {'$unwind': '$complaints'},
    {'$group': {
        '_id': '$customer.customer_id',
        'home_zone': {'$first': '$customer.home_zone'},
        'n_complaints': {'$sum': 1},
        'total_compensation': {'$sum': '$complaints.compensation_amount'},
        'avg_resolution': {'$avg': '$complaints.resolution_days'},
        'last_complaint_ts': {'$last': '$complaints.created_at'}
    }},
    {'$match': {'n_complaints': {'$gte': 2}}},
    {'$sort': {'total_compensation': -1}},
    {'$limit': 10}
]
for d in cases.aggregate(p2): print(d)

In [ ]:
# Pipeline 3 — hub incident pressure with $lookup join
# (requires the hubs collection — load it first)
hubs_df = pd.read_csv('hubs.csv')
db['hubs'].drop()
db['hubs'].insert_many(hubs_df.to_dict('records'))

p3 = [
    {'$unwind': '$incidents'},
    {'$match': {'incidents.severity': {'$in': ['High', 'Critical']}}},
    {'$group': {
        '_id': '$delivery.hub_id',
        'n_high_inc': {'$sum': 1},
        'n_failures': {'$sum': {
            '$cond': [{'$eq': ['$delivery.status', 'Failed']}, 1, 0]}}
    }},
    {'$lookup': {'from': 'hubs', 'localField': '_id',
                 'foreignField': 'hub_id', 'as': 'hub_info'}},
    {'$unwind': '$hub_info'},
    {'$project': {
        'hub_name': '$hub_info.hub_name',
        'hub_type': '$hub_info.hub_type',
        'capacity': '$hub_info.capacity_score',
        'n_high_inc': 1, 'n_failures': 1
    }},
    {'$sort': {'n_high_inc': -1}}
]
for d in cases.aggregate(p3): print(d)

## 8. Verify all collections

In [ ]:
for name in db.list_collection_names():
    n = db[name].count_documents({})
    print(f'  {name:20s}: {n:5d} documents')